In [ ]:
from google.colab import files
import os
import zipfile


uploaded = files.upload()

# Create kaggle directory and move the file
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d dartweichen/student-life

# Unzip the dataset
with zipfile.ZipFile('student-life.zip', 'r') as zip_ref:
    zip_ref.extractall('student-life-data')

# List the contents
import os
print("Dataset contents:")
for item in os.listdir('student-life-data'):
    print(f" - {item}")

In [ ]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
         os.path.join(dirname, filename)

In [ ]:
import warnings
import glob
from functools import reduce
warnings.filterwarnings('ignore')

<h1> UTILITIES</h1>

In [ ]:
def clean_columns(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    return df

def clean_timestamp(df, timestamp_col='timestamp'):
    # Convert timestamp to datetime
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], unit="s", utc=True)
    df['date'] = df[timestamp_col].dt.date
    return df

def read_student_files(base_path, prefix):
    files = glob.glob(os.path.join(base_path, f"{prefix}*.csv"))
    return [(os.path.basename(f).split("_")[1].split(".")[0], f) for f in files]

def concat_summaries(summaries):
    if summaries:
        return pd.concat(summaries, ignore_index=True)
    else:
        return pd.DataFrame()


<h1>ACTIVITY MODULE</h1>

In [ ]:
def get_daily_activity_summary(df):
    daily_activity = (
    df
    .groupby(['user_id', 'date'])['activity_inference']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reset_index()
)

    return daily_activity


In [ ]:
def process_activity(path, student):
    df = pd.read_csv(path)
    df = clean_columns(df)
    df = clean_timestamp(df, 'timestamp')
    df['user_id'] = student
    activity_map = {
    0: 'Stationary',
    1: 'Walking',
    2: 'Running',
    3: 'Unknown'
}
    df['activity_inference'] = df['activity_inference'].map(activity_map)

    return get_daily_activity_summary(df)

<h1>GPS MODULE</h1>

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Compute great-circle distance (in km) between two points given latitude & longitude.
    """
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c


In [ ]:
def compute_gps_daily_metrics(df):
    gps_daily = df.groupby('date').agg(
        dist_total_km=('dist_km', 'sum'),
        dist_mean_km=('dist_km', 'mean'),
        dist_std_km=('dist_km', 'std'),
        lat_std=('latitude', 'std'),
        lon_std=('longitude', 'std')
    ).reset_index()

    home_lat, home_lon = df['latitude'].mode()[0], df['longitude'].mode()[0]
    df['dist_from_home_km'] = haversine(df['latitude'], df['longitude'], home_lat, home_lon)

    home_time = (
        df[df['dist_from_home_km'] < 0.2]
        .groupby('date').size().reset_index(name='home_points')
    )

    network_mode = df.groupby('date')['network_type'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None).reset_index()

    gps_daily = (
        gps_daily.merge(home_time, on='date', how='left')
                  .merge(network_mode, on='date', how='left')
                  .fillna({'home_points': 0})
    )

    return gps_daily


In [ ]:
def process_gps(path, student):
    df = pd.read_csv(path,index_col=False)
    df = clean_columns(df)
    df = df.drop(["accuracy","bearing","altitude","speed","travelstate"], axis=1, errors='ignore')
    df = clean_timestamp(df, 'time')
    df = df.sort_values(['date','time']).reset_index(drop=True)
    df['dist_km'] = haversine(df['latitude'], df['longitude'],
                              df['latitude'].shift(), df['longitude'].shift()).fillna(0)
    gps_summary = compute_gps_daily_metrics(df)
    gps_summary['user_id'] = student
    return gps_summary

<h1> CONVERSATION MODULE</h1>

In [ ]:
def conversation_metrics(df):
    return df.groupby('date').agg(
        conversation_count=('duration_minutes', 'count'),
        total_conversation_minutes=('duration_minutes', 'sum'),
        avg_conversation_minutes=('duration_minutes', 'mean')
    ).reset_index()

def process_conversation(path, student):
    df = pd.read_csv(path)
    df = clean_columns(df)
    df = clean_timestamp(df, 'start_timestamp')
    df = clean_timestamp(df, 'end_timestamp')
    df['duration_minutes'] = (df['end_timestamp'] - df['start_timestamp']).dt.total_seconds() / 60
    df = df.sort_values('date').reset_index(drop=True)
    con_summary = conversation_metrics(df)
    con_summary['user_id'] = student
    return con_summary

<h1>BLUETOOTH MODULE</h1>

In [ ]:
def blue_metrics(df):
    return df.groupby('date').agg(
        avg_bt_level=('level', 'mean'),
        num_bt_devices=('mac', 'nunique')
    ).reset_index()

def process_bluetooth(path, student):
    df = pd.read_csv(path)
    df = clean_columns(df)
    df = clean_timestamp(df, 'time')
    df = df.sort_values('date').reset_index(drop=True)
    blue_summary = blue_metrics(df)
    blue_summary['user_id'] = student
    return blue_summary

<h1>PIPLINE Per User</h1>

In [ ]:
def run_pipeline_per_user():
    base_paths = {
        "activity": "/content/student-life-data/dataset/sensing/activity/",
        "gps": "/content/student-life-data/dataset/sensing/gps/",
        "conversation": "/content/student-life-data/dataset/sensing/conversation/",
        "bt": "/content/student-life-data/dataset/sensing/bluetooth/"
    }

    process_funcs = {
        "activity": process_activity,
        "gps": process_gps,
        "conversation": process_conversation,
        "bt": process_bluetooth
    }

    # Get list of all users from one modality (assuming same naming)
    students = [os.path.basename(f).split("_")[1].split(".")[0]
                for f in glob.glob(os.path.join(base_paths["activity"], "activity_*.csv"))]

    all_users = []

    for student in students:
        dfs = []

        for module, func in process_funcs.items():
            path = os.path.join(base_paths[module], f"{module}_{student}.csv")
            if os.path.exists(path):
                try:
                    dfs.append(func(path, student))
                except Exception as e:
                    print(f"Skipped {module} for {student}: {e}")

        # Merge modalities per student
        if dfs:
            merged_user = reduce(lambda left, right: pd.merge(left, right, on=['user_id', 'date'], how='outer'), dfs)
            merged_user['user_id'] = student
            all_users.append(merged_user)

    return concat_summaries(all_users)


In [ ]:
merged_all_users = run_pipeline_per_user()

# Merge with sleep
sleep = pd.read_csv('/content/sleep.csv')
sleep['date'] = pd.to_datetime(sleep['date'], errors='coerce').dt.date
sleep['user_id'] = sleep['user_id'].astype(str)

merged_final = pd.merge(merged_all_users, sleep, on=['user_id', 'date'], how='outer')
merged_final = merged_final.sort_values(['user_id', 'date']).reset_index(drop=True)

print("Final shape:", merged_final.shape)
merged_final.head()

Final shape: (2903, 20)


,user_id,date,Running,Stationary,Unknown,Walking,dist_total_km,dist_mean_km,dist_std_km,lat_std,lon_std,home_points,network_type,conversation_count,total_conversation_minutes,avg_conversation_minutes,avg_bt_level,num_bt_devices,sleep_duration,sleep_quality
0,u00,2013-03-27,0.046536,0.812952,0.030422,0.110090,65.395914,1.127516,3.644554,0.030746,0.029574,9.0,wifi,21.0,382.850000,18.230952,-84.810651,39.0,8.0,2.0
1,u00,2013-03-28,0.022662,0.910270,0.017029,0.050039,167.400792,2.461776,5.293769,0.043453,0.052639,5.0,wifi,21.0,348.366667,16.588889,-85.340000,37.0,6.5,2.0
2,u00,2013-03-29,0.013046,0.838184,0.034370,0.114400,110.812228,1.704804,4.238148,0.029443,0.031415,9.0,wifi,32.0,479.116667,14.972396,-85.989796,56.0,6.0,2.0
3,u00,2013-03-30,0.006839,0.885659,0.031844,0.075657,20.411229,0.497835,1.747863,0.027030,0.009952,27.0,wifi,31.0,431.366667,13.915054,-80.916667,6.0,NaN,NaN
4,u00,2013-03-31,0.076462,0.808346,0.024613,0.090580,83.016935,1.153013,3.684254,0.022226,0.021175,65.0,wifi,23.0,694.350000,30.189130,NaN,NaN,9.0,2.0


In [ ]:
merged_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2903 entries, 0 to 2902
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   user_id                     2903 non-null   object 
 1   date                        2903 non-null   object 
 2   Running                     2883 non-null   float64
 3   Stationary                  2883 non-null   float64
 4   Unknown                     2883 non-null   float64
 5   Walking                     2883 non-null   float64
 6   dist_total_km               2824 non-null   float64
 7   dist_mean_km                2824 non-null   float64
 8   dist_std_km                 2801 non-null   float64
 9   lat_std                     2801 non-null   float64
 10  lon_std                     2801 non-null   float64
 11  home_points                 2824 non-null   float64
 12  network_type                2793 non-null   object 
 13  conversation_count          2729 

In [ ]:
merged_final['user_id'].nunique()

49

In [ ]:
merged_final['date'].nunique()

67

In [ ]:
merged_final['date'].min() , merged_final['date'].max()

(datetime.date(2013, 3, 27), datetime.date(2013, 6, 1))

In [ ]:
# for i in range(60):
#     print(i,daily_activity.loc[daily_activity.user_id==f'u{str(i).zfill(2)}'].shape)

Since student u39 has only 24 responses instead of 67 across all sensors, we can deduce that this user is the additional participant (likely a stakeholder or test user). Therefore, we will exclude this user from further analysis.

In [ ]:
merged_final = merged_final.loc[merged_final['user_id'] != 'u39']

<h1>Mapping User Ids</h1>

In [ ]:
def mapping(df):
    unique_ids = sorted(df['user_id'].unique())
    new_ids = [f"u{str(i).zfill(2)}" for i in range(len(unique_ids))]
    id_map = dict(zip(unique_ids, new_ids))
    df['user_id'] = df['user_id'].map(id_map)
    print(df['user_id'].unique())
    print("Total users:", df['user_id'].nunique())

mapping(merged_final)

['u00' 'u01' 'u02' 'u03' 'u04' 'u05' 'u06' 'u07' 'u08' 'u09' 'u10' 'u11'
 'u12' 'u13' 'u14' 'u15' 'u16' 'u17' 'u18' 'u19' 'u20' 'u21' 'u22' 'u23'
 'u24' 'u25' 'u26' 'u27' 'u28' 'u29' 'u30' 'u31' 'u32' 'u33' 'u34' 'u35'
 'u36' 'u37' 'u38' 'u39' 'u40' 'u41' 'u42' 'u43' 'u44' 'u45' 'u46' 'u47']
Total users: 48


### Create the master date DataFrame

In [ ]:
start_date = '2013-03-27'
end_date   = '2013-06-01'
dates = pd.date_range(start=start_date, end=end_date)
dates_df = pd.DataFrame({'date': dates})

ids = [f"u{str(i).zfill(2)}" for i in range(48)]
ids_df   = pd.DataFrame({'user_id': ids})

all_users_dates = ids_df.merge(dates_df, how='cross')

print(all_users_dates.shape)   #  (3216, 2)

(3216, 2)


In [ ]:
all_users_dates['date'].nunique(), all_users_dates['user_id'].nunique()

(67, 48)

# Merge all dataframes

In [ ]:
merged_final['date']=pd.to_datetime(merged_final['date'])
merged_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2879 entries, 0 to 2902
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   user_id                     2879 non-null   object        
 1   date                        2879 non-null   datetime64[ns]
 2   Running                     2859 non-null   float64       
 3   Stationary                  2859 non-null   float64       
 4   Unknown                     2859 non-null   float64       
 5   Walking                     2859 non-null   float64       
 6   dist_total_km               2800 non-null   float64       
 7   dist_mean_km                2800 non-null   float64       
 8   dist_std_km                 2777 non-null   float64       
 9   lat_std                     2777 non-null   float64       
 10  lon_std                     2777 non-null   float64       
 11  home_points                 2800 non-null   float64       
 1

In [ ]:
all_users_dates['user_id'].unique()

array(['u00', 'u01', 'u02', 'u03', 'u04', 'u05', 'u06', 'u07', 'u08',
       'u09', 'u10', 'u11', 'u12', 'u13', 'u14', 'u15', 'u16', 'u17',
       'u18', 'u19', 'u20', 'u21', 'u22', 'u23', 'u24', 'u25', 'u26',
       'u27', 'u28', 'u29', 'u30', 'u31', 'u32', 'u33', 'u34', 'u35',
       'u36', 'u37', 'u38', 'u39', 'u40', 'u41', 'u42', 'u43', 'u44',
       'u45', 'u46', 'u47'], dtype=object)

In [ ]:
trial = all_users_dates.merge(merged_final, how='left', on=['user_id', 'date'])
trial.head()

,user_id,date,Running,Stationary,Unknown,Walking,dist_total_km,dist_mean_km,dist_std_km,lat_std,lon_std,home_points,network_type,conversation_count,total_conversation_minutes,avg_conversation_minutes,avg_bt_level,num_bt_devices,sleep_duration,sleep_quality
0,u00,2013-03-27,0.046536,0.812952,0.030422,0.110090,65.395914,1.127516,3.644554,0.030746,0.029574,9.0,wifi,21.0,382.850000,18.230952,-84.810651,39.0,8.0,2.0
1,u00,2013-03-28,0.022662,0.910270,0.017029,0.050039,167.400792,2.461776,5.293769,0.043453,0.052639,5.0,wifi,21.0,348.366667,16.588889,-85.340000,37.0,6.5,2.0
2,u00,2013-03-29,0.013046,0.838184,0.034370,0.114400,110.812228,1.704804,4.238148,0.029443,0.031415,9.0,wifi,32.0,479.116667,14.972396,-85.989796,56.0,6.0,2.0
3,u00,2013-03-30,0.006839,0.885659,0.031844,0.075657,20.411229,0.497835,1.747863,0.027030,0.009952,27.0,wifi,31.0,431.366667,13.915054,-80.916667,6.0,NaN,NaN
4,u00,2013-03-31,0.076462,0.808346,0.024613,0.090580,83.016935,1.153013,3.684254,0.022226,0.021175,65.0,wifi,23.0,694.350000,30.189130,NaN,NaN,9.0,2.0


In [ ]:
trial.shape

(3216, 20)

In [ ]:
trial.isna().sum()

,0
user_id,0
date,0
Running,357
Stationary,357
Unknown,357
Walking,357
dist_total_km,416
dist_mean_km,416
dist_std_km,439
lat_std,439


In [ ]:
trial.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3216 entries, 0 to 3215
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   user_id                     3216 non-null   object        
 1   date                        3216 non-null   datetime64[ns]
 2   Running                     2859 non-null   float64       
 3   Stationary                  2859 non-null   float64       
 4   Unknown                     2859 non-null   float64       
 5   Walking                     2859 non-null   float64       
 6   dist_total_km               2800 non-null   float64       
 7   dist_mean_km                2800 non-null   float64       
 8   dist_std_km                 2777 non-null   float64       
 9   lat_std                     2777 non-null   float64       
 10  lon_std                     2777 non-null   float64       
 11  home_points                 2800 non-null   float64     

In [ ]:
trial=trial.drop(['network_type'],axis=1)

In [ ]:
def fill_nulls(df):
  # Sort by user_id and date
    df = df.sort_values(by=['user_id', 'date'])

    # df = df.set_index('date')

    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

    df[numeric_cols] = df.groupby('user_id')[numeric_cols].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )

    # Reset index to get date back as column
    # df = df.reset_index()

    return df


In [ ]:
final_trial=fill_nulls(trial)

In [ ]:
final_trial.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3216 entries, 0 to 3215
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   user_id                     3216 non-null   object        
 1   date                        3216 non-null   datetime64[ns]
 2   Running                     3216 non-null   float64       
 3   Stationary                  3216 non-null   float64       
 4   Unknown                     3216 non-null   float64       
 5   Walking                     3216 non-null   float64       
 6   dist_total_km               3216 non-null   float64       
 7   dist_mean_km                3216 non-null   float64       
 8   dist_std_km                 3216 non-null   float64       
 9   lat_std                     3216 non-null   float64       
 10  lon_std                     3216 non-null   float64       
 11  home_points                 3216 non-null   float64     

In [ ]:
final_trial.describe().T

,count,mean,min,25%,50%,75%,max,std
date,3216,2013-04-29 00:00:00,2013-03-27 00:00:00,2013-04-12 00:00:00,2013-04-29 00:00:00,2013-05-16 00:00:00,2013-06-01 00:00:00,NaN
Running,3216.0,0.010148,0.0,0.000124,0.003754,0.01204,1.0,0.038543
Stationary,3216.0,0.922465,0.0,0.898858,0.939344,0.966949,1.0,0.079325
Unknown,3216.0,0.026612,0.0,0.005877,0.013939,0.028927,0.518306,0.043543
Walking,3216.0,0.040775,0.0,0.016534,0.033663,0.055053,0.827586,0.038549
dist_total_km,3216.0,40.343822,0.001597,1.51126,3.460538,8.914591,6884.038367,271.604129
dist_mean_km,3216.0,2.832411,0.000064,0.025199,0.053058,0.148936,1399.948782,40.470819
dist_std_km,3216.0,5.983977,0.000128,0.077927,0.138502,0.530861,2423.214327,72.123484
lat_std,3216.0,0.02535,0.0,0.000668,0.001356,0.006042,3.915492,0.149435
lon_std,3216.0,0.035441,0.0,0.000829,0.001709,0.004644,22.188876,0.477101


In [ ]:
final_trial['user_id'].nunique()

48

In [ ]:
final_trial['date'].nunique()

67

In [ ]:
final_trial.shape

(3216, 19)

In [ ]:
final_trial.head()

,user_id,date,Running,Stationary,Unknown,Walking,dist_total_km,dist_mean_km,dist_std_km,lat_std,lon_std,home_points,conversation_count,total_conversation_minutes,avg_conversation_minutes,avg_bt_level,num_bt_devices,sleep_duration,sleep_quality
0,u00,2013-03-27,0.046536,0.812952,0.030422,0.110090,65.395914,1.127516,3.644554,0.030746,0.029574,9.0,21.0,382.850000,18.230952,-84.810651,39.0,8.0,2.0
1,u00,2013-03-28,0.022662,0.910270,0.017029,0.050039,167.400792,2.461776,5.293769,0.043453,0.052639,5.0,21.0,348.366667,16.588889,-85.340000,37.0,6.5,2.0
2,u00,2013-03-29,0.013046,0.838184,0.034370,0.114400,110.812228,1.704804,4.238148,0.029443,0.031415,9.0,32.0,479.116667,14.972396,-85.989796,56.0,6.0,2.0
3,u00,2013-03-30,0.006839,0.885659,0.031844,0.075657,20.411229,0.497835,1.747863,0.027030,0.009952,27.0,31.0,431.366667,13.915054,-80.916667,6.0,7.5,2.0
4,u00,2013-03-31,0.076462,0.808346,0.024613,0.090580,83.016935,1.153013,3.684254,0.022226,0.021175,65.0,23.0,694.350000,30.189130,-82.485279,27.0,9.0,2.0


In [ ]:
# final_trial.to_csv("final_merge.csv")

# Personal baseline

In [ ]:
baseline_df = final_trial.copy()

In [ ]:
baseline_df = baseline_df.sort_values(['user_id','date'])

In [ ]:
behavior_cols = [
    'sleep_duration',
    'dist_total_km',
    'conversation_count',
    'total_conversation_minutes',
    'num_bt_devices',
    'Stationary',
    'Walking',
    'Running'
]

In [ ]:
window = 10

for col in behavior_cols:

    baseline_df[f'{col}_mean'] = (
        baseline_df.groupby('user_id')[col]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=5).mean())
    )

    baseline_df[f'{col}_std'] = (
        baseline_df.groupby('user_id')[col]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=5).std())
    )


In [ ]:
for col in behavior_cols:
    baseline_df[f'{col}_z'] = (
        (baseline_df[col] - baseline_df[f'{col}_mean']) /
        (baseline_df[f'{col}_std'] + 1e-6)
    )

In [ ]:
baseline_df.head(6)

,user_id,date,Running,Stationary,Unknown,Walking,dist_total_km,dist_mean_km,dist_std_km,lat_std,...,Running_mean,Running_std,sleep_duration_z,dist_total_km_z,conversation_count_z,total_conversation_minutes_z,num_bt_devices_z,Stationary_z,Walking_z,Running_z
0,u00,2013-03-27,0.046536,0.812952,0.030422,0.110090,65.395914,1.127516,3.644554,0.030746,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,u00,2013-03-28,0.022662,0.910270,0.017029,0.050039,167.400792,2.461776,5.293769,0.043453,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,u00,2013-03-29,0.013046,0.838184,0.034370,0.114400,110.812228,1.704804,4.238148,0.029443,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,u00,2013-03-30,0.006839,0.885659,0.031844,0.075657,20.411229,0.497835,1.747863,0.027030,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,u00,2013-03-31,0.076462,0.808346,0.024613,0.090580,83.016935,1.153013,3.684254,0.022226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,u00,2013-04-01,0.014352,0.907026,0.014976,0.063647,10.729613,0.149022,0.677181,0.026397,...,0.033109,0.028555,-3.685911,-1.441327,0.25646,0.178271,0.817709,1.239699,-0.92875,-0.656858


In [ ]:
baseline_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3216 entries, 0 to 3215
Data columns (total 43 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   user_id                          3216 non-null   object        
 1   date                             3216 non-null   datetime64[ns]
 2   Running                          3216 non-null   float64       
 3   Stationary                       3216 non-null   float64       
 4   Unknown                          3216 non-null   float64       
 5   Walking                          3216 non-null   float64       
 6   dist_total_km                    3216 non-null   float64       
 7   dist_mean_km                     3216 non-null   float64       
 8   dist_std_km                      3216 non-null   float64       
 9   lat_std                          3216 non-null   float64       
 10  lon_std                          3216 non-null   float64    

In [ ]:
baseline_df.drop(columns=['sleep_duration_mean',
       'sleep_duration_std', 'dist_total_km_mean', 'dist_total_km_std',
       'conversation_count_mean', 'conversation_count_std',
       'total_conversation_minutes_mean', 'total_conversation_minutes_std',
       'num_bt_devices_mean', 'num_bt_devices_std', 'Stationary_mean',
       'Stationary_std', 'Walking_mean', 'Walking_std', 'Running_mean',
       'Running_std'],inplace=True)

In [ ]:
baseline_df.to_csv('baseline_behavior.csv')